In [ ]:
# ==========================================================
# S-Learner + IPTW + Kalibrasyon (TR) — JSON dışa aktarma ile
# Gerekenler: numpy, pandas, scikit-learn, xgboost
# pip install numpy pandas scikit-learn xgboost
# ==========================================================
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from IPython.display import display
import json

# ------------------- AYAR -------------------
CSV_PATH     = "final_dataset_with_complication_types_risk_added_with_names.csv"
QUERY_ID     = 136          # <- patient_id değeri (kesin eşleşme)
CALIB_METHOD = "isotonic"  # "isotonic" | "sigmoid"
VAL_SIZE     = 0.30
SEED         = 42
np.random.seed(SEED)

# ------------------- VERİYİ YÜKLE -------------------
df = df = pd.read_csv(CSV_PATH, encoding="ISO-8859-9")  # Türkçe için ideal

# Yaş metinse sayıya çevir
if df['age'].dtype == 'O':
    age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
    df['age'] = df['age'].map(age_map).fillna(df['age']).astype(float)

# io_group yoksa io_type'tan üret; o da yoksa Monotherapy
def to_group(io_type):
    mapping = {
        'PD-1 monotherapy': 'Monotherapy',
        'PD-L1 monotherapy': 'Monotherapy',
        'Chemo + PD-1': 'Chemo_IO',
        'Chemo + PD-L1': 'Chemo_IO',
        'CTLA-4 + PD-1 combo': 'Other',
        'IO-contraindicated': 'Other'
    }
    return mapping.get(str(io_type), 'Monotherapy')

if 'io_group' not in df.columns:
    if 'io_type' in df.columns:
        df['io_group'] = df['io_type'].apply(to_group)
    else:
        df['io_group'] = 'Monotherapy'

# ------------------- KOLON KONTROLÜ -------------------
req = ['pd_l1','tmb','age','msi','metastasis','ecog_score',
       'smoking_status','cancer_stage','treatment_start_days',
       'response','io_group','complication_type','complication_risk_pct']
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Eksik kolon(lar): {missing}")

# Temizlik
df = df.dropna(subset=[c for c in req if c != 'complication_risk_pct']).copy()
df['response']          = df['response'].astype(int)
df['io_group']          = df['io_group'].astype(str)
df['complication_type'] = df['complication_type'].astype(str)

# patient_id: 1..N (sabit) — JSON eşleştirme için
df = df.reset_index(drop=True).copy()
df['patient_id'] = np.arange(1, len(df) + 1).astype(int)

print("Tedavi grupları:", df['io_group'].unique().tolist())
print("Sınıf dengesi (response=1 oranı):", round(df['response'].mean(), 3))
print(f"{df['complication_type'].nunique()} komplikasyon tipi bulundu.")

# ------------------- ÖZELLİKLER -------------------
features_num   = ['pd_l1','tmb','age','ecog_score','treatment_start_days']
features_cat   = ['msi','metastasis','smoking_status','cancer_stage']
base_features  = features_num + features_cat
target         = "response"
treatment_col  = "io_group"

# ------------------- PROPENSITY (IPTW) -------------------
df['io_group'] = df['io_group'].fillna('Monotherapy').astype(str)
print("Tedavi dağılımı:")
print(df['io_group'].value_counts(dropna=False))

prop_preproc = ColumnTransformer([
    ("num", StandardScaler(), features_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat)
])
X_prop = prop_preproc.fit_transform(df[base_features])

le_treat = LabelEncoder()
y_treat_raw     = df[treatment_col].astype(str).values
y_prop_enc      = le_treat.fit_transform(y_treat_raw)
present_classes = le_treat.classes_
print("Veride bulunan tedavi sınıfları:", list(present_classes))

ALL_TREAT_CLASSES = np.array(sorted(set(list(present_classes) + ["Monotherapy","Chemo_IO","Other"])))
skip_propensity = (len(present_classes) < 2)

if skip_propensity:
    print("⚠️ Uyarı: Veride tek tedavi sınıfı var; IPTW atlanıyor (tüm ağırlıklar=1).")
    df['_t_idx'] = 0
    df['_p_t']   = 1.0
    w_iptw       = np.ones(len(df), dtype=np.float32)
else:
    prop_model = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        n_estimators=400,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        n_jobs=-1
    )
    try:
        prop_model.fit(X_prop, y_prop_enc)
        P_all = prop_model.predict_proba(X_prop)
        df['_t_idx'] = le_treat.transform(df[treatment_col].astype(str))
        df['_p_t']   = P_all[np.arange(len(df)), df['_t_idx']].clip(1e-6, 1-1e-6)
        w_iptw       = (1.0 / df['_p_t']).astype(np.float32).values
        q_hi = np.quantile(w_iptw, 0.99)               # stabilize weights
        w_iptw = np.minimum(w_iptw, q_hi).astype(np.float32)
    except Exception as e:
        print("⚠️ Propensity eğitimi başarısız oldu, IPTW atlanıyor. Sebep:", repr(e))
        df['_t_idx'] = 0
        df['_p_t']   = 1.0
        w_iptw       = np.ones(len(df), dtype=np.float32)

TREAT_CLASSES_FOR_EVAL = ALL_TREAT_CLASSES.copy()

# ------------------- S-LEARNER -------------------
s_features_cat = features_cat + [treatment_col]
s_preproc = ColumnTransformer([
    ("num", StandardScaler(), features_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), s_features_cat)
])

X_all = df[base_features + [treatment_col]].copy()
y_all = df[target].values.astype(int)
X_all_proc = s_preproc.fit_transform(X_all).astype(np.float32)

X_tr, X_va, y_tr, y_va, w_tr, w_va = train_test_split(
    X_all_proc, y_all, w_iptw, test_size=VAL_SIZE, random_state=SEED,
    stratify=df[treatment_col]
)

base_outcome = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=500,
    max_depth=4,
    learning_rate=0.06,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    n_jobs=-1
)
base_outcome.fit(X_tr, y_tr, sample_weight=w_tr)

# Kalibrasyon
try:
    cal_outcome = CalibratedClassifierCV(estimator=base_outcome, method=CALIB_METHOD, cv='prefit')
except TypeError:
    cal_outcome = CalibratedClassifierCV(base_estimator=base_outcome, method=CALIB_METHOD, cv='prefit')

try:
    cal_outcome.fit(X_va, y_va, sample_weight=w_va)
except TypeError:
    cal_outcome.fit(X_va, y_va)

y_va_pred = cal_outcome.predict_proba(X_va)[:,1]
avg_true = float(y_va.mean())
avg_pred = float(y_va_pred.mean()) if y_va_pred.mean() > 1e-9 else 1.0
CORRECTION = avg_true / avg_pred
print("\n✅ Outcome (S-Learner) eğitildi + kalibre edildi.")
print("Val ort. tahmin (kalibre):", round(y_va_pred.mean(), 3), "| Gerçek ort.:", round(y_va.mean(), 3))
print("Post-hoc düzeltme katsayısı:", round(CORRECTION, 3))

# ------------------- YARDIMCI FONKSİYONLAR -------------------
def _adjust(p: float) -> float:
    return float(np.clip(p * CORRECTION, 0.0, 1.0))

def _get_row_by_pid(pid: int):
    """Sadece patient_id eşleşmesi. Eşleşme yoksa hata."""
    pid_int = int(pid)
    hit = df.loc[df['patient_id'] == pid_int]
    if hit.empty:
        raise ValueError(f"patient_id={pid_int} için kayıt bulunamadı.")
    return hit.iloc[0]

def _print_patient(row):
    print("\n🧾 Seçilen kayıt:")
    print(f"  • patient_id: {int(row['patient_id'])}")
    if 'full_name' in row:
        print(f"  • full_name: {row['full_name']}")
    for c in base_features:
        print(f"  • {c}: {row[c]}")

def predict_by_id(pid: int):
    row = _get_row_by_pid(pid)
    _print_patient(row)

    base_df = pd.DataFrame([{k: row[k] for k in base_features}])

    rows = []
    best_treat, best_prob = None, -1.0
    for t in TREAT_CLASSES_FOR_EVAL:
        cf = base_df.copy()
        cf[treatment_col] = t
        Xp = s_preproc.transform(cf).astype(np.float32)
        p  = float(cal_outcome.predict_proba(Xp)[0, 1])
        p  = _adjust(p)
        rows.append((t, round(p*100, 2)))
        if p > best_prob:
            best_prob, best_treat = p, t

    res_treat = (pd.DataFrame(rows, columns=["Tedavi","Başarı_%"])
                 .sort_values("Başarı_%", ascending=False)
                 .reset_index(drop=True))

    print("\n🔮 Tedavi başarı tahminleri:")
    try:
        display(res_treat)  # varsa Jupyter/Colab
    except Exception:
        print(res_treat.to_string(index=False))
    print(f"👉 En uygun tedavi: {res_treat.iloc[0,0]} (≈ %{res_treat.iloc[0,1]:.2f})")

    ctype = row['complication_type']
    if 'complication_risk_pct' in row and pd.notna(row['complication_risk_pct']):
        crisk = float(row['complication_risk_pct'])
        print("\n⚠️ En olası komplikasyon:")
        print(f"Komplikasyon: {ctype} (Risk ≈ %{crisk:.2f})")
    elif 'complication_grade' in row and pd.notna(row.get('complication_grade', np.nan)):
        g = float(row['complication_grade'])
        g_norm = g if 0 <= g <= 1 else np.clip(g/4.0, 0, 1)
        label = "düşük" if g_norm < 0.33 else ("orta" if g_norm < 0.66 else "yüksek")
        print("\n⚠️ En olası komplikasyon:")
        print(f"Komplikasyon: {ctype} (Şiddet: {label}, derece≈{g})")
    else:
        print("\n⚠️ Komplikasyon riski bilgisi yok.")

    if 'complication_top3' in row and isinstance(row['complication_top3'], str):
        print("\nℹ️ Bu hasta için model tabanlı en olası ilk 3 komplikasyon (referans):")
        print(row['complication_top3'])

    return res_treat

def export_prediction_json(pid: int, out_path: str):
    """Seçilen hasta için iPad uygulamasının okuyacağı JSON’u yazar."""
    row = _get_row_by_pid(pid)
    base_df = pd.DataFrame([{k: row[k] for k in base_features}])

    best_treat, best_prob = None, -1.0
    alternatives = []
    for t in TREAT_CLASSES_FOR_EVAL:
        cf = base_df.copy()
        cf[treatment_col] = t
        Xp = s_preproc.transform(cf).astype(np.float32)
        p  = float(cal_outcome.predict_proba(Xp)[0, 1])
        p  = _adjust(p)
        alternatives.append({"treatment": t, "success_probability": round(p, 4)})
        if p > best_prob:
            best_prob, best_treat = p, t

    # Küçük bir sahte “history” (grafik için)
    now = pd.Timestamp.utcnow()
    history = []
    for i in range(10):
        history.append({
            "timestamp": (now - pd.Timedelta(minutes=5*(9-i))).isoformat(),
            "success_probability": round(max(0.0, min(1.0, best_prob - 0.03 + 0.006*i)), 4)
        })

    data = {
        "patient_id": int(row["patient_id"]),
        "full_name": (row["full_name"] if "full_name" in row else None),
        "recommended_treatment": best_treat,
        "current_probability": round(float(best_prob), 4),
        "alternatives": alternatives,
        "complication_type": str(row["complication_type"]),
        "complication_risk_pct": (
            float(row["complication_risk_pct"]) if pd.notna(row.get("complication_risk_pct", np.nan)) else None
        ),
        "history": history
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON yazıldı → {out_path}")

# ------------------- ÖRNEK KULLANIM -------------------
if __name__ == "__main__":
    # Konsolda sonucu gör
    _ = predict_by_id(QUERY_ID)

    # iPad uygulaması için JSON üret
    export_prediction_json(QUERY_ID, f"patient_{QUERY_ID}_prediction.json")

In [ ]:
# service.py
import numpy as np, pandas as pd, warnings, json
warnings.filterwarnings("ignore")

from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# ----- AYARLAR -----
CSV_PATH     = "final_dataset_with_complication_types_risk_added_with_names.csv"
CALIB_METHOD = "isotonic"
VAL_SIZE     = 0.30
SEED         = 42
np.random.seed(SEED)

# ----- VERİ/MODELİ YÜKLE & EĞİT -----
df = df = pd.read_csv(CSV_PATH, encoding="ISO-8859-9")  # Türkçe için ideal

# Yaş metinse sayıya çevir
if df['age'].dtype == 'O':
    age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
    df['age'] = df['age'].map(age_map).fillna(df['age']).astype(float)

def to_group(io_type):
    mapping = {
        'PD-1 monotherapy': 'Monotherapy',
        'PD-L1 monotherapy': 'Monotherapy',
        'Chemo + PD-1': 'Chemo_IO',
        'Chemo + PD-L1': 'Chemo_IO',
        'CTLA-4 + PD-1 combo': 'Other',
        'IO-contraindicated': 'Other'
    }
    return mapping.get(str(io_type), 'Monotherapy')

if 'io_group' not in df.columns:
    if 'io_type' in df.columns:
        df['io_group'] = df['io_type'].apply(to_group)
    else:
        df['io_group'] = 'Monotherapy'

req = ['pd_l1','tmb','age','msi','metastasis','ecog_score',
       'smoking_status','cancer_stage','treatment_start_days',
       'response','io_group','complication_type','complication_risk_pct']
missing = [c for c in req if c not in df.columns]
if missing:
    raise RuntimeError(f"Eksik kolon(lar): {missing}")

df = df.dropna(subset=[c for c in req if c != 'complication_risk_pct']).copy()
df['response']          = df['response'].astype(int)
df['io_group']          = df['io_group'].astype(str)
df['complication_type'] = df['complication_type'].astype(str)

df = df.reset_index(drop=True).copy()
df['patient_id'] = np.arange(1, len(df) + 1).astype(int)

features_num   = ['pd_l1','tmb','age','ecog_score','treatment_start_days']
features_cat   = ['msi','metastasis','smoking_status','cancer_stage']
base_features  = features_num + features_cat
target         = "response"
treatment_col  = "io_group"

df['io_group'] = df['io_group'].fillna('Monotherapy').astype(str)

prop_preproc = ColumnTransformer([
    ("num", StandardScaler(), features_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat)
])
X_prop = prop_preproc.fit_transform(df[base_features])

le_treat = LabelEncoder()
y_treat_raw     = df[treatment_col].astype(str).values
y_prop_enc      = le_treat.fit_transform(y_treat_raw)
present_classes = le_treat.classes_
ALL_TREAT_CLASSES = np.array(sorted(set(list(present_classes) + ["Monotherapy","Chemo_IO","Other"])))
skip_propensity = (len(present_classes) < 2)

if skip_propensity:
    w_iptw = np.ones(len(df), dtype=np.float32)
else:
    prop_model = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        n_estimators=400,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        n_jobs=-1
    )
    prop_model.fit(X_prop, y_prop_enc)
    P_all = prop_model.predict_proba(X_prop)
    t_idx = le_treat.transform(df[treatment_col].astype(str))
    p_t   = P_all[np.arange(len(df)), t_idx].clip(1e-6, 1-1e-6)
    w_iptw = (1.0 / p_t).astype(np.float32)
    q_hi = np.quantile(w_iptw, 0.99)
    w_iptw = np.minimum(w_iptw, q_hi).astype(np.float32)

s_features_cat = features_cat + [treatment_col]
s_preproc = ColumnTransformer([
    ("num", StandardScaler(), features_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), s_features_cat)
])
X_all = df[base_features + [treatment_col]].copy()
y_all = df[target].values.astype(int)
X_all_proc = s_preproc.fit_transform(X_all).astype(np.float32)

from sklearn.model_selection import train_test_split
X_tr, X_va, y_tr, y_va, w_tr, w_va = train_test_split(
    X_all_proc, y_all, w_iptw, test_size=VAL_SIZE, random_state=SEED,
    stratify=df[treatment_col]
)

base_outcome = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=500,
    max_depth=4,
    learning_rate=0.06,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    n_jobs=-1
)
base_outcome.fit(X_tr, y_tr, sample_weight=w_tr)

try:
    cal_outcome = CalibratedClassifierCV(estimator=base_outcome, method=CALIB_METHOD, cv='prefit')
except TypeError:
    cal_outcome = CalibratedClassifierCV(base_estimator=base_outcome, method=CALIB_METHOD, cv='prefit')

try:
    cal_outcome.fit(X_va, y_va, sample_weight=w_va)
except TypeError:
    cal_outcome.fit(X_va, y_va)

y_va_pred = cal_outcome.predict_proba(X_va)[:,1]
avg_true = float(y_va.mean())
avg_pred = float(y_va_pred.mean()) if y_va_pred.mean() > 1e-9 else 1.0
CORRECTION = avg_true / avg_pred

def _adjust(p: float) -> float:
    return float(np.clip(p * CORRECTION, 0.0, 1.0))

def _row_by_pid(pid: int):
    hit = df.loc[df['patient_id'] == int(pid)]
    if hit.empty:
        raise KeyError(f"patient_id={pid} bulunamadı")
    return hit.iloc[0]

def _predict_payload(pid: int):
    row = _row_by_pid(pid)
    base_df = pd.DataFrame([{k: row[k] for k in base_features}])

    best_treat, best_prob = None, -1.0
    alternatives = []
    for t in ALL_TREAT_CLASSES:
        cf = base_df.copy()
        cf[treatment_col] = t
        Xp = s_preproc.transform(cf).astype(np.float32)
        p  = float(cal_outcome.predict_proba(Xp)[0, 1])
        p  = _adjust(p)
        alternatives.append({"treatment": t, "success_probability": round(p, 4)})
        if p > best_prob:
            best_prob, best_treat = p, t

    # küçük history (grafik için)
    now = pd.Timestamp.utcnow()
    history = []
    for i in range(10):
        history.append({
            "timestamp": (now - pd.Timedelta(minutes=5*(9-i))).isoformat(),
            "success_probability": round(max(0.0, min(1.0, best_prob - 0.03 + 0.006*i)), 4)
        })

    return {
        "patient_id": int(row["patient_id"]),
        "full_name": (row["full_name"] if "full_name" in row else None),
        "recommended_treatment": best_treat,
        "current_probability": round(float(best_prob), 4),
        "alternatives": alternatives,
        "complication_type": str(row["complication_type"]),
        "complication_risk_pct": (
            float(row["complication_risk_pct"]) if pd.notna(row.get("complication_risk_pct", np.nan)) else None
        ),
        "history": history
    }

# ----- FASTAPI -----
app = FastAPI(title="OncoAI API", version="1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # geliştirmede açık; prod’da kısıtla
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/search")
def search(name: str = Query(..., description="Hasta adı (parça eşleşme)")):
    name_l = name.strip().lower()
    if not name_l:
        return []
    # basit contains taraması
    if "full_name" not in df.columns:
        raise HTTPException(500, "Veri setinde full_name yok.")
    m = df["full_name"].astype(str).str.lower().str.contains(name_l, na=False)
    res = df.loc[m, ["patient_id","full_name","age","cancer_stage","ecog_score"]].copy()
    res = res.rename(columns={"patient_id":"id","cancer_stage":"stage","ecog_score":"ecogScore"})
    res["id"] = res["id"].astype(int)
    # UI için cancerType alanı (yoksa boş string)
    res["cancerType"] = df.loc[m, "cancer_type"] if "cancer_type" in df.columns else ""
    # dizi döndür
    return res.to_dict(orient="records")

@app.get("/predict")
def predict(patient_id: int = Query(..., description="patient_id")):
    try:
        payload = _predict_payload(patient_id)
        return payload
    except KeyError as e:
        raise HTTPException(404, str(e))